<a href="https://colab.research.google.com/github/AshokGit544/Enterprise-Finance-Decision-Copilot1.1/blob/main/Enterprise_Finance_Decision_Copilot1_1_.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# ============================================================
# STEP 1: INSTALL PACKAGES
# ============================================================

!pip -q install pandas numpy faker scikit-learn sentence-transformers pyarrow openpyxl

In [ ]:
# ============================================================
# STEP 2: IMPORTS AND PROJECT FOLDERS
# ============================================================

import os
import json
import random
from pathlib import Path
from datetime import datetime, timedelta

import numpy as np
import pandas as pd
from faker import Faker

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
fake = Faker()
Faker.seed(SEED)

PROJECT_NAME = "enterprise-finance-decision-copilot"
PROJECT_DIR = Path(PROJECT_NAME)

DATA_DIR = PROJECT_DIR / "data"
RAW_DIR = DATA_DIR / "raw"
OUTPUT_DIR = DATA_DIR / "output"
CONFIG_DIR = PROJECT_DIR / "config"

for folder in [RAW_DIR, OUTPUT_DIR, CONFIG_DIR]:
    folder.mkdir(parents=True, exist_ok=True)

print("Project folders created successfully.")
print("Project path:", PROJECT_DIR.resolve())

In [ ]:
# ============================================================
# STEP 3: PROJECT CONFIG
# ============================================================

config = {
    "project_name": "Enterprise Finance Decision Copilot",
    "domain": "SAP FICO style finance intelligence",
    "objective": "Generate enterprise finance data, run DQ checks, build decision logic, create semantic search and decision copilot output.",
    "version": "1.0.0",
    "author": "Ranjith",
    "created_at": datetime.now().isoformat()
}

with open(CONFIG_DIR / "project_config.json", "w", encoding="utf-8") as f:
    json.dump(config, f, indent=4)

print("Config file saved successfully.")
print(json.dumps(config, indent=2))

In [ ]:
# ============================================================
# STEP 4: CREATE MASTER DATA
# ============================================================

vendor_categories = [
    "Maintenance Services",
    "IT Services",
    "Electrical Components",
    "Field Operations",
    "Logistics",
    "Consulting",
    "Inspection Services"
]

vendor_states = ["NY", "NJ", "PA", "CT", "MA", "MI", "IL", "TX"]

gl_accounts = pd.DataFrame([
    ["400100", "Revenue - Energy Delivery", "Revenue"],
    ["400200", "Revenue - Grid Services", "Revenue"],
    ["500100", "Opex - Maintenance", "Expense"],
    ["500200", "Opex - IT Services", "Expense"],
    ["500300", "Opex - Contractors", "Expense"],
    ["500400", "Opex - Logistics", "Expense"],
    ["500500", "Opex - Compliance", "Expense"],
    ["110100", "Accounts Payable", "Liability"]
], columns=["gl_account", "gl_account_name", "gl_type"])

cost_centers = pd.DataFrame([
    ["CC100", "Transmission Operations", "Operations"],
    ["CC110", "Distribution Maintenance", "Operations"],
    ["CC120", "Substation Engineering", "Engineering"],
    ["CC130", "Customer Support", "Customer Operations"],
    ["CC140", "Finance Operations", "Finance"],
    ["CC150", "Supplier Quality", "Quality"],
    ["CC160", "Asset Reliability", "Reliability"]
], columns=["cost_center_id", "cost_center_name", "department"])

invoice_descriptions = [
    "transformer repair service",
    "sap fico support work",
    "field contractor invoice",
    "substation maintenance work",
    "it platform support",
    "logistics support for equipment",
    "distribution network inspection",
    "compliance audit service",
    "urgent material replacement",
    "vendor maintenance support"
]

policy_topics = [
    "invoice approval threshold",
    "duplicate invoice control",
    "late payment escalation",
    "vendor compliance review",
    "high amount approval workflow",
    "unmatched purchase order handling"
]

print("Master data references ready.")

In [ ]:
# ============================================================
# STEP 5: GENERATE VENDOR MASTER DATA
# ============================================================

vendor_rows = []

for i in range(1, 51):
    vendor_id = f"V{i:03d}"
    vendor_name = fake.company()
    vendor_category = random.choice(vendor_categories)
    vendor_state = random.choice(vendor_states)
    vendor_risk_score = round(random.uniform(0.05, 0.95), 2)
    preferred_vendor_flag = random.choice(["Y", "N"])

    vendor_rows.append([
        vendor_id,
        vendor_name,
        vendor_category,
        vendor_state,
        vendor_risk_score,
        preferred_vendor_flag
    ])

vendors = pd.DataFrame(vendor_rows, columns=[
    "vendor_id",
    "vendor_name",
    "vendor_category",
    "vendor_state",
    "vendor_risk_score",
    "preferred_vendor_flag"
])

vendors.to_csv(RAW_DIR / "vendors.csv", index=False)
gl_accounts.to_csv(RAW_DIR / "gl_accounts.csv", index=False)
cost_centers.to_csv(RAW_DIR / "cost_centers.csv", index=False)

print("Master data saved.")
display(vendors.head())

In [ ]:
# ============================================================
# STEP 6: GENERATE TRANSACTION AND POLICY DATA
# ============================================================

start_date = datetime(2024, 1, 1)

invoice_rows = []
payment_rows = []
po_rows = []
policy_rows = []
case_rows = []

approval_statuses = ["Approved", "Pending", "Rejected", "Under Review"]
payment_statuses = ["Paid", "Partially Paid", "Pending"]
invoice_sources = ["SAP", "Ariba", "Coupa", "Manual Upload"]
business_units = ["Utility Finance", "Grid Ops", "Shared Services", "Procurement"]

# Purchase Orders
for i in range(1, 121):
    po_id = f"PO{i:05d}"
    vendor = vendors.sample(1).iloc[0]
    cc = cost_centers.sample(1).iloc[0]
    po_date = start_date + timedelta(days=random.randint(0, 365))
    po_amount = round(random.uniform(5000, 150000), 2)

    po_rows.append([
        po_id,
        vendor["vendor_id"],
        cc["cost_center_id"],
        po_date.date().isoformat(),
        po_amount,
        random.choice(["Open", "Closed", "Released"]),
        random.choice(business_units)
    ])

purchase_orders = pd.DataFrame(po_rows, columns=[
    "po_id", "vendor_id", "cost_center_id", "po_date",
    "po_amount_usd", "po_status", "business_unit"
])

# Invoices and Payments
for i in range(1, 301):
    vendor = vendors.sample(1).iloc[0]
    gl = gl_accounts[gl_accounts["gl_type"].isin(["Expense", "Revenue"])].sample(1).iloc[0]
    cc = cost_centers.sample(1).iloc[0]
    po = purchase_orders.sample(1).iloc[0]

    posting_date = start_date + timedelta(days=random.randint(0, 365))
    invoice_date = posting_date - timedelta(days=random.randint(1, 15))
    due_date = posting_date + timedelta(days=random.randint(15, 45))
    amount = round(random.uniform(1000, 85000), 2)

    invoice_id = f"INV{i:05d}"
    source_doc_id = f"SRC{i:05d}"

    invoice_rows.append([
        invoice_id,
        source_doc_id,
        po["po_id"],
        vendor["vendor_id"],
        gl["gl_account"],
        cc["cost_center_id"],
        invoice_date.date().isoformat(),
        posting_date.date().isoformat(),
        due_date.date().isoformat(),
        amount,
        "USD",
        random.choice(invoice_descriptions),
        random.choice(approval_statuses),
        random.choice(invoice_sources),
        random.choice(business_units)
    ])

    payment_status = random.choice(payment_statuses)
    if payment_status == "Paid":
        paid_amount = round(amount * random.uniform(0.97, 1.00), 2)
        payment_date = due_date + timedelta(days=random.randint(-5, 20))
    elif payment_status == "Partially Paid":
        paid_amount = round(amount * random.uniform(0.30, 0.80), 2)
        payment_date = due_date + timedelta(days=random.randint(0, 25))
    else:
        paid_amount = 0.0
        payment_date = due_date + timedelta(days=random.randint(10, 35))

    payment_rows.append([
        f"PAY{i:05d}",
        source_doc_id,
        invoice_id,
        payment_date.date().isoformat(),
        paid_amount,
        payment_status
    ])

invoices = pd.DataFrame(invoice_rows, columns=[
    "invoice_id", "source_doc_id", "po_id", "vendor_id", "gl_account",
    "cost_center_id", "invoice_date", "posting_date", "due_date",
    "amount_usd", "currency", "description", "approval_status",
    "invoice_source", "business_unit"
])

payments = pd.DataFrame(payment_rows, columns=[
    "payment_id", "source_doc_id", "invoice_id",
    "payment_date", "paid_amount_usd", "payment_status"
])

# Policies
for i in range(1, 31):
    topic = random.choice(policy_topics)
    threshold = random.choice([5000, 10000, 25000, 50000])

    policy_rows.append([
        f"POL{i:03d}",
        topic,
        f"{topic.title()} policy requires validation for invoices above {threshold} USD and mandatory review if vendor risk is high or PO match is missing.",
        random.choice(["Finance Controls", "Procurement Governance", "AP Operations"]),
        random.choice(["Low", "Medium", "High"])
    ])

policies = pd.DataFrame(policy_rows, columns=[
    "policy_id", "policy_topic", "policy_text", "policy_owner", "severity"
])

# Investigation Cases
case_reasons = [
    "Duplicate invoice suspicion",
    "High-risk vendor review",
    "Late payment follow-up",
    "Missing PO validation",
    "Amount threshold escalation",
    "Compliance exception check"
]

for i in range(1, 81):
    case_rows.append([
        f"CASE{i:04d}",
        random.choice(case_reasons),
        random.choice(invoices["invoice_id"].tolist()),
        random.choice(["Open", "Closed", "In Progress"]),
        random.choice(["Low", "Medium", "High"]),
        fake.sentence(nb_words=10)
    ])

investigation_cases = pd.DataFrame(case_rows, columns=[
    "case_id", "case_reason", "invoice_id", "case_status", "case_priority", "case_summary"
])

purchase_orders.to_csv(RAW_DIR / "purchase_orders.csv", index=False)
invoices.to_csv(RAW_DIR / "invoices.csv", index=False)
payments.to_csv(RAW_DIR / "payments.csv", index=False)
policies.to_csv(RAW_DIR / "policies.csv", index=False)
investigation_cases.to_csv(RAW_DIR / "investigation_cases.csv", index=False)

print("Transaction and policy datasets saved.")
display(invoices.head())

In [ ]:
# ============================================================
# STEP 7: INJECT REALISTIC DATA ISSUES
# ============================================================

invoices.loc[3, "vendor_id"] = None
invoices.loc[7, "amount_usd"] = -500.00
invoices.loc[10, "description"] = None
invoices.loc[15, "amount_usd"] = 175000.00
invoices.loc[15, "approval_status"] = "Pending"
invoices.loc[20, ["vendor_id", "gl_account", "cost_center_id", "amount_usd", "description"]] = \
    invoices.loc[21, ["vendor_id", "gl_account", "cost_center_id", "amount_usd", "description"]].values
invoices.loc[25, "po_id"] = None

payments.loc[30, "payment_status"] = "Pending"
payments.loc[30, "paid_amount_usd"] = 0.0

invoices.to_csv(RAW_DIR / "invoices.csv", index=False)
payments.to_csv(RAW_DIR / "payments.csv", index=False)

print("Issues injected successfully.")

In [ ]:
# ============================================================
# STEP 8: LOAD RAW FILES AND BUILD INTEGRATED VIEW
# ============================================================

vendors_df = pd.read_csv(RAW_DIR / "vendors.csv")
gl_accounts_df = pd.read_csv(RAW_DIR / "gl_accounts.csv")
cost_centers_df = pd.read_csv(RAW_DIR / "cost_centers.csv")
purchase_orders_df = pd.read_csv(RAW_DIR / "purchase_orders.csv")
invoices_df = pd.read_csv(RAW_DIR / "invoices.csv")
payments_df = pd.read_csv(RAW_DIR / "payments.csv")
policies_df = pd.read_csv(RAW_DIR / "policies.csv")
cases_df = pd.read_csv(RAW_DIR / "investigation_cases.csv")

integrated_df = (
    invoices_df
    .merge(vendors_df, on="vendor_id", how="left")
    .merge(gl_accounts_df, on="gl_account", how="left")
    .merge(cost_centers_df, on="cost_center_id", how="left")
    .merge(purchase_orders_df[["po_id", "po_amount_usd", "po_status"]], on="po_id", how="left")
    .merge(payments_df, on=["invoice_id", "source_doc_id"], how="left")
)

print("Integrated dataframe created.")
print("Shape:", integrated_df.shape)
display(integrated_df.head())

In [ ]:
# ============================================================
# STEP 9: STANDARDIZATION AND COMMON BUSINESS KEY
# ============================================================

integrated_df["vendor_name_clean"] = integrated_df["vendor_name"].fillna("").str.strip().str.upper()
integrated_df["description_clean"] = integrated_df["description"].fillna("").str.strip().str.upper()
integrated_df["cost_center_name_clean"] = integrated_df["cost_center_name"].fillna("").str.strip().str.upper()
integrated_df["gl_account_name_clean"] = integrated_df["gl_account_name"].fillna("").str.strip().str.upper()

integrated_df["common_business_key"] = (
    integrated_df["vendor_id"].fillna("MISSING_VENDOR").astype(str) + "_" +
    integrated_df["cost_center_id"].fillna("MISSING_CC").astype(str) + "_" +
    integrated_df["gl_account"].fillna("MISSING_GL").astype(str)
)

print("Common business key generated.")
display(integrated_df[["invoice_id", "common_business_key"]].head())

In [ ]:
# ============================================================
# STEP 10: DATA QUALITY CHECKS
# ============================================================

dq_records = []

for _, row in integrated_df.iterrows():
    if pd.isna(row["vendor_id"]):
        dq_records.append([row["invoice_id"], "Missing vendor_id"])
    if pd.notna(row["amount_usd"]) and float(row["amount_usd"]) <= 0:
        dq_records.append([row["invoice_id"], "Invalid amount"])
    if pd.isna(row["description"]) or str(row["description"]).strip() == "":
        dq_records.append([row["invoice_id"], "Missing description"])
    if pd.isna(row["po_id"]):
        dq_records.append([row["invoice_id"], "Missing po_id"])

data_quality_issues = pd.DataFrame(dq_records, columns=["invoice_id", "issue_reason"]).drop_duplicates()

print("Data quality checks completed.")
display(data_quality_issues.head(20))

In [ ]:
# ============================================================
# STEP 11: RISK FLAGS AND DECISION LOGIC
# ============================================================

today_ref = pd.Timestamp("2025-01-31")

integrated_df["invoice_date"] = pd.to_datetime(integrated_df["invoice_date"], errors="coerce")
integrated_df["posting_date"] = pd.to_datetime(integrated_df["posting_date"], errors="coerce")
integrated_df["due_date"] = pd.to_datetime(integrated_df["due_date"], errors="coerce")
integrated_df["payment_date"] = pd.to_datetime(integrated_df["payment_date"], errors="coerce")

integrated_df["days_past_due"] = (today_ref - integrated_df["due_date"]).dt.days
integrated_df["unpaid_flag"] = np.where(integrated_df["payment_status"] == "Pending", "Y", "N")
integrated_df["high_amount_flag"] = np.where(integrated_df["amount_usd"] > 50000, "Y", "N")
integrated_df["high_vendor_risk_flag"] = np.where(integrated_df["vendor_risk_score"] > 0.75, "Y", "N")
integrated_df["late_payment_flag"] = np.where(integrated_df["days_past_due"] > 15, "Y", "N")
integrated_df["po_missing_flag"] = np.where(integrated_df["po_id"].isna(), "Y", "N")

dup_cols = ["vendor_id", "gl_account", "cost_center_id", "amount_usd", "description"]
integrated_df["duplicate_invoice_flag"] = np.where(
    integrated_df.duplicated(subset=dup_cols, keep=False), "Y", "N"
)

def decide_action(row):
    reasons = []

    if row["high_amount_flag"] == "Y":
        reasons.append("high invoice amount")
    if row["high_vendor_risk_flag"] == "Y":
        reasons.append("high vendor risk")
    if row["late_payment_flag"] == "Y":
        reasons.append("late payment")
    if row["po_missing_flag"] == "Y":
        reasons.append("missing PO")
    if row["duplicate_invoice_flag"] == "Y":
        reasons.append("possible duplicate invoice")
    if row["unpaid_flag"] == "Y":
        reasons.append("still unpaid")

    if len(reasons) >= 3:
        return "Escalate for finance control review", ", ".join(reasons)
    elif len(reasons) == 2:
        return "Send for analyst investigation", ", ".join(reasons)
    elif len(reasons) == 1:
        return "Monitor and validate", ", ".join(reasons)
    else:
        return "No immediate action needed", "no major risk signals"

decision_outputs = integrated_df.apply(lambda row: decide_action(row), axis=1, result_type="expand")
integrated_df["recommended_action"] = decision_outputs[0]
integrated_df["decision_reason"] = decision_outputs[1]

print("Risk flags and decisions created.")
display(integrated_df[[
    "invoice_id", "amount_usd", "payment_status",
    "high_amount_flag", "high_vendor_risk_flag",
    "late_payment_flag", "po_missing_flag",
    "duplicate_invoice_flag", "recommended_action", "decision_reason"
]].head(15))

In [ ]:
# ============================================================
# STEP 12: CREATE RETRIEVAL-READY TEXT
# ============================================================

integrated_df["retrieval_text"] = (
    "Invoice " + integrated_df["invoice_id"].astype(str) +
    " Vendor " + integrated_df["vendor_name"].fillna("Unknown Vendor").astype(str) +
    " Cost Center " + integrated_df["cost_center_name"].fillna("Unknown Cost Center").astype(str) +
    " GL Account " + integrated_df["gl_account_name"].fillna("Unknown GL").astype(str) +
    " Amount " + integrated_df["amount_usd"].astype(str) +
    " Description " + integrated_df["description"].fillna("No Description").astype(str) +
    " Approval Status " + integrated_df["approval_status"].fillna("Unknown").astype(str) +
    " Payment Status " + integrated_df["payment_status"].fillna("Unknown").astype(str) +
    " Recommended Action " + integrated_df["recommended_action"].astype(str) +
    " Decision Reason " + integrated_df["decision_reason"].astype(str)
)

retrieval_ready_documents = integrated_df[[
    "invoice_id", "common_business_key", "retrieval_text"
]].copy()

embedding_input = retrieval_ready_documents.copy()

print("Retrieval-ready documents created.")
display(retrieval_ready_documents.head())

In [ ]:
# ============================================================
# STEP 13: CREATE POLICY RETRIEVAL TEXT
# ============================================================

policies_df["retrieval_text"] = (
    "Policy ID " + policies_df["policy_id"].astype(str) +
    " Topic " + policies_df["policy_topic"].astype(str) +
    " Owner " + policies_df["policy_owner"].astype(str) +
    " Severity " + policies_df["severity"].astype(str) +
    " Policy Text " + policies_df["policy_text"].astype(str)
)

policy_retrieval_docs = policies_df[["policy_id", "policy_topic", "retrieval_text"]].copy()

print("Policy retrieval docs created.")
display(policy_retrieval_docs.head())

In [ ]:
# ============================================================
# STEP 14: SAVE OUTPUT FILES
# ============================================================

integrated_df.to_csv(OUTPUT_DIR / "integrated_finance_view.csv", index=False)
data_quality_issues.to_csv(OUTPUT_DIR / "data_quality_issues.csv", index=False)
retrieval_ready_documents.to_csv(OUTPUT_DIR / "retrieval_ready_documents.csv", index=False)
embedding_input.to_csv(OUTPUT_DIR / "embedding_input.csv", index=False)
policy_retrieval_docs.to_csv(OUTPUT_DIR / "policy_retrieval_documents.csv", index=False)

decision_summary = integrated_df[[
    "invoice_id", "common_business_key", "recommended_action", "decision_reason",
    "high_amount_flag", "high_vendor_risk_flag", "late_payment_flag",
    "po_missing_flag", "duplicate_invoice_flag"
]].copy()

decision_summary.to_csv(OUTPUT_DIR / "decision_summary.csv", index=False)

print("Output files saved successfully.")
for p in sorted(OUTPUT_DIR.glob("*")):
    print("-", p.name)

In [ ]:
# ============================================================
# STEP 15: LOAD EMBEDDING MODEL (NO TOKEN NEEDED)
# ============================================================

from sentence_transformers import SentenceTransformer

EMBEDDING_MODEL_NAME = "sentence-transformers/all-MiniLM-L6-v2"
embedding_model = SentenceTransformer(EMBEDDING_MODEL_NAME)

print("Embedding model loaded successfully.")
print("Model:", EMBEDDING_MODEL_NAME)

In [25]:
# ============================================================
# STEP 16: CREATE INVOICE EMBEDDINGS
# ============================================================

invoice_texts = retrieval_ready_documents["retrieval_text"].fillna("").tolist()

invoice_embeddings = embedding_model.encode(
    invoice_texts,
    show_progress_bar=True,
    convert_to_numpy=True
)

print("Invoice embeddings created.")
print("Shape:", invoice_embeddings.shape)

Batches:   0%|          | 0/10 [00:00<?, ?it/s]

Invoice embeddings created.
Shape: (300, 384)


In [26]:
# ============================================================
# STEP 17: CREATE POLICY EMBEDDINGS
# ============================================================

policy_texts = policy_retrieval_docs["retrieval_text"].fillna("").tolist()

policy_embeddings = embedding_model.encode(
    policy_texts,
    show_progress_bar=True,
    convert_to_numpy=True
)

print("Policy embeddings created.")
print("Shape:", policy_embeddings.shape)

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Policy embeddings created.
Shape: (30, 384)


In [27]:
# ============================================================
# STEP 18: SAVE EMBEDDINGS
# ============================================================

np.save(OUTPUT_DIR / "invoice_embeddings.npy", invoice_embeddings)
np.save(OUTPUT_DIR / "policy_embeddings.npy", policy_embeddings)

print("Embeddings saved successfully.")

Embeddings saved successfully.


In [28]:
# ============================================================
# STEP 19: SEMANTIC SEARCH FUNCTION
# ============================================================

from sklearn.metrics.pairwise import cosine_similarity

def semantic_search(query, documents_df, embeddings, model, top_k=5):
    query_embedding = model.encode([query], convert_to_numpy=True)
    scores = cosine_similarity(query_embedding, embeddings)[0]

    result_df = documents_df.copy()
    result_df["similarity_score"] = scores
    result_df = result_df.sort_values(by="similarity_score", ascending=False).head(top_k)

    return result_df

In [29]:
# ============================================================
# STEP 20: TEST SEMANTIC SEARCH
# ============================================================

query_1 = "show me high risk invoices with missing purchase order and late payment"

invoice_search_results = semantic_search(
    query=query_1,
    documents_df=retrieval_ready_documents,
    embeddings=invoice_embeddings,
    model=embedding_model,
    top_k=5
)

print("Top invoice search results:")
display(invoice_search_results)

query_2 = "what policy applies to high amount invoice approval and missing po"

policy_search_results = semantic_search(
    query=query_2,
    documents_df=policy_retrieval_docs,
    embeddings=policy_embeddings,
    model=embedding_model,
    top_k=5
)

print("Top policy search results:")
display(policy_search_results)

Top invoice search results:


,invoice_id,common_business_key,retrieval_text,similarity_score
225,INV00226,V019_CC160_500200,"Invoice INV00226 Vendor Adams, Zuniga and Wong...",0.506420
181,INV00182,V039_CC130_500400,Invoice INV00182 Vendor Harrell LLC Cost Cente...,0.503801
277,INV00278,V044_CC120_500400,Invoice INV00278 Vendor Allen-Allen Cost Cente...,0.501325
98,INV00099,V044_CC160_400200,Invoice INV00099 Vendor Allen-Allen Cost Cente...,0.497727
58,INV00059,V047_CC160_500300,Invoice INV00059 Vendor Anderson Group Cost Ce...,0.497197


Top policy search results:


,policy_id,policy_topic,retrieval_text,similarity_score
28,POL029,invoice approval threshold,Policy ID POL029 Topic invoice approval thresh...,0.704264
18,POL019,duplicate invoice control,Policy ID POL019 Topic duplicate invoice contr...,0.684132
12,POL013,high amount approval workflow,Policy ID POL013 Topic high amount approval wo...,0.681710
27,POL028,high amount approval workflow,Policy ID POL028 Topic high amount approval wo...,0.669628
6,POL007,invoice approval threshold,Policy ID POL007 Topic invoice approval thresh...,0.666206
